# Ch 12 Networked Programs

Chapter 12 of Py4E gets into networked programs, primarily using HyperText Transfer Protocol (http).

## Sockets

This chapter introduces the concept of a **socket**. This is something that will continue to come up, so is important to understand. 

In many ways, a socket is like a file handle--it provides access to the information, not the information itself. However, a socket is different in that it provides **two-way** communication for sending *and* receiving information.

Here, we'll use sockets to connect to a web server and get the contents of a web page. Different from opening a file on disk, more coordination is needed between your computer and the web server to transmit data, confirm receipt of data, etc. Later, we'll use sockets to connect to databases. And again, coordination and established protocols for sending and receiving data come into play.

## Protocols

As described in the text, for two computers to communicate successfully, they need to be following some protocol, or established procedures for communicating. HTTP is one protocol. We looked briefly at the SFTP (Secure File Transfer Protocol) earlier in the semester for transferring files from our computers to the cluster. Other protocols you may be familiar with include Internet Message Access Protocol (IMAP), Post Office Protocol version 3 (POP3) and Simple Mail Transfer Protocol (SMTP) all used for email systems.

There are many protocols for different types of communications, the important thing is that you need to establish which protocol is being used and follow the specifications of that protocol.

## 12.2 The world’s simplest web browser (p. 146)

Here's the code for `socket1.py` (remember these are in the code3 directory of the repository).

In [ ]:
import socket

mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
mysock.connect(('data.pr4e.org', 80))
cmd = 'GET http://data.pr4e.org/romeo.txt HTTP/1.0\r\n\r\n'.encode()
mysock.send(cmd)

while True:
    data = mysock.recv(512) 
    if (len(data) < 1):
        break
    print(data.decode(),end='')

mysock.close()

# Code: http://www.py4e.com/code3/socket1.py

Note that if you navigate to [http://data.pr4e.org/romeo.txt](http://data.pr4e.org/romeo.txt) in your browser, you see the content, but not the headers--your web browser uses the headers to understand the content and format it correctly for you.

## 12.3 Retrieving an image of HTTP (p. 148)

Take a look at the `urljpeg.py` script, which downloads an image, and the information on the script in the text.


In [ ]:
import socket
import time

HOST = 'data.pr4e.org'
PORT = 80
mysock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
mysock.connect((HOST, PORT))
mysock.sendall(b'GET http://data.pr4e.org/cover3.jpg HTTP/1.0\r\n\r\n')
count = 0
picture = b""

while True:
    data = mysock.recv(5120)
    if len(data) < 1: break
    #time.sleep(0.25)
    count = count + len(data)
    print(len(data), count)
    picture = picture + data

mysock.close()

# Look for the end of the header (2 CRLF)
pos = picture.find(b"\r\n\r\n")
print('Header length', pos)
print(picture[:pos].decode())

# Skip past the header and save the picture data
picture = picture[pos+4:]
fhand = open("stuff.jpg", "wb")
fhand.write(picture)
fhand.close()

# Code: http://www.py4e.com/code3/urljpeg.py

Hopefully most of this makes some sense, but starts looking kind of messy...The next sections will show you that while you *can* work at this low-level and write code to speak directly to the remote server, there are modules to simplify this. But fundamentally, they provide an easier user experience to the same functionality.


## 12.4 Retrieving web pages with `urllib`


The `urllib` module makes getting stuff from the web a bit easier. The `socket1.py` script above can be simplified to:

In [ ]:
import urllib.request
fhand = urllib.request.urlopen('http://data.pr4e.org/romeo.txt') 
for line in fhand:
    print(line.decode().strip())
# Code: http://www.py4e.com/code3/urllib1.py

Notice that `urllib` sits between the script and the socket to make an opened socket look just like a file handle and we can treat the web page in much the same way as we treat a local file.

## 12.5 Reading binary files using `urllib`

And the rather messy `urljpg.py` to download an image is simplified to:

In [3]:
import urllib.request, urllib.parse, urllib.error
img = urllib.request.urlopen('http://data.pr4e.org/cover3.jpg').read()
fhand = open('cover3.jpg', 'wb')
fhand.write(img)
fhand.close()
# Code: http://www.py4e.com/code3/curl1.py

The chapter goes on to show how to write out chunks of the file so as not to accumulate everything in RAM. 

## 12.7 Parsing HTML using regular expressions (p. 152)

We could write our own scripts to parse HTML looking for information. The example here is using a regular expression to search for a link and make a list of links on a page.

Run `urlregex.py` on some site, google.com for example:

```bash
[magitz@login8 code3]$ python3 urlregex.py
Enter - http://google.com
http://www.google.com/imghp?hl=en&tab=wi
http://maps.google.com/maps?hl=en&tab=wl
https://play.google.com/?hl=en&tab=w8
http://www.youtube.com/?gl=US&tab=w1
http://news.google.com/nwshp?hl=en&tab=wn
https://mail.google.com/mail/?tab=wm
https://drive.google.com/?tab=wo
https://www.google.com/intl/en/options/
http://www.google.com/history/optout?hl=en
https://accounts.google.com/ServiceLogin?hl=en&passive=true&continue=http://www.google.com/
https://plus.google.com/116899029375914044550
[magitz@login8 code3]$ 
```

Unfortunately, not all web pages follow HTML guidelines totally and sometimes pages can be really hard to parse. Tags can be upper and lower case, some closing tags are optional, etc. It can quickly get quite complex.

## 12.8 Parsing HTML using BeautifulSoup

As I mentioned earlier, whatever you are trying to do, look for a module to make your life easier. If you need to parse HTML, don't start trying to write your own script to do it, look at available modules. One is `BeautifulSoup` available from crummy.com--some people sure have fun naming things!


In [ ]:
import urllib.request, urllib.parse, urllib.error
from bs4 import BeautifulSoup
import ssl

# Ignore SSL certificate errors
ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

url = input('Enter - ')
html = urllib.request.urlopen(url, context=ctx).read()
soup = BeautifulSoup(html, 'html.parser')

# Retrieve all of the anchor tags
tags = soup('a') 
for tag in tags:
    print(tag.get('href', None))

# Code: http://www.py4e.com/code3/urllinks.py


## Where to go from here?

There's lots of data on the web and lots of our world lives online..Explore options and think about data sources you are interested in.

The next chapter goes beyond getting information from web pages to interacting with networked applications using API (Application Programming Interfaces)